In [ ]:

import pandas as pd
import numpy as np

from scipy.stats import spearmanr, zscore
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf


In [ ]:
expr = pd.read_csv(
    "data/OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv",
    index_col=0
)

models = pd.read_csv("data/Model.csv")

prism = pd.read_csv(
    "data/secondary-screen-dose-response-curve-parameters.csv"
)

treatments = pd.read_csv(
    "data/secondary-screen-replicate-collapsed-treatment-info.csv"
)

cell_info = pd.read_csv(
    "data/secondary-screen-cell-line-info.csv"
)

In [ ]:
def get_drug_response(
    prism,
    requested_drug,
    response_col="auc",
):
    drug_df = prism.loc[
        prism["name"].astype(str).str.lower()
        == requested_drug.lower(),
        ["depmap_id", "name", response_col],
    ].copy()

    drug_df[response_col] = pd.to_numeric(
        drug_df[response_col],
        errors="coerce",
    )

    drug_df = drug_df.dropna(
        subset=["depmap_id", response_col]
    )

    
    drug_df = (
        drug_df
        .groupby("depmap_id", as_index=False)
        .agg(
            name=("name", "first"),
            **{response_col: (response_col, "median")},
        )
    )

    return drug_df

In [ ]:



treatment_hypotheses = {
    "CDK4/6": {
        "targets": ["CDK4", "CDK6"],
        "drugs": ["palbociclib", "ribociclib", "abemaciclib"],
    },
    "WEE1": {
        "targets": ["WEE1"],
        "drugs": ["MK-1775", "AZD1775", "adavosertib", "azenosertib", "ZN-c3", "Debio 0123"],
    },
    "CHK1": {
        "targets": ["CHEK1"],
        "drugs": ["LY2603618", "SCH-900776", "MK-8776", "prexasertib"],
    },
    "SLC7A11/ferroptosis": {
        "targets": ["SLC7A11"],
        "drugs": [
            "sulfasalazine", "erastin", "imidazole ketone erastin",
            "RSL3", "ML162", "FIN56", "FINO2"
        ],
    },
    "PI3K": {
        "targets": ["PIK3CA", "PIK3R1", "PIK3R3"],
        "drugs": ["alpelisib", "pictilisib", "buparlisib", "taselisib", "copanlisib"],
    },
    "AKT": {
        "targets": ["AKT1", "AKT2", "AKT3"],
        "drugs": ["capivasertib", "ipatasertib", "MK-2206"],
    },
    "mTOR": {
        "targets": ["MTOR", "RPTOR"],
        "drugs": [
            "everolimus", "temsirolimus", "sirolimus", "ridaforolimus",
            "AZD8055", "Torin 1", "Torin 2"
        ],
    },
}


drug_annotation = {
    drug.lower(): hypothesis
    for hypothesis, config in treatment_hypotheses.items()
    for drug in config["drugs"]
}

requested_drugs = [
    drug
    for config in treatment_hypotheses.values()
    for drug in config["drugs"]
]


def gene_symbol(col):
    return str(col).split(" ")[0].upper()


expr = expr.rename(columns=lambda col: gene_symbol(col))


In [ ]:

prism["name_lower"] = prism["name"].astype(str).str.lower()

available_names = set(prism["name_lower"].dropna())

drug_availability = pd.DataFrame({
    "hypothesis": [drug_annotation[d.lower()] for d in requested_drugs],
    "requested_drug": requested_drugs,
    "available_in_PRISM": [d.lower() in available_names for d in requested_drugs],
})

drug_availability.sort_values(["hypothesis", "requested_drug"])


In [ ]:
print("Expression matrix:", expr.shape)
print("PRISM rows:", len(prism))
print("Available requested drugs:", drug_availability["available_in_PRISM"].sum())


In [ ]:
def build_mss_tmbh_score(
    expr,
    model_ids,
    de,
    gene_col="gene",
    lfc_col="log2FoldChange",
    padj_col="padj",
    padj_cutoff=0.05,
    lfc_cutoff=1,
    top_n=50,
    exclude_genes=None,
):
    """Build an MSS/TMB-H transcriptional similarity score.

    Score = mean(z-scored upregulated genes) - mean(z-scored downregulated genes)
    Higher values indicate greater similarity to the TCGA MSS/TMB-H profile.
    """
    exclude_genes = {str(g).upper() for g in (exclude_genes or [])}

    sig = de[
        (de[padj_col] < padj_cutoff) &
        (de[lfc_col].abs() >= lfc_cutoff)
    ].copy()
    sig[gene_col] = sig[gene_col].astype(str).str.upper()
    sig = sig[~sig[gene_col].isin(exclude_genes)]

    
    up = sig[sig[lfc_col] < 0].sort_values(lfc_col, ascending=False).head(top_n)
    down = sig[sig[lfc_col] > 0].sort_values(lfc_col, ascending=True).head(top_n)

    up_genes = up[gene_col].tolist()
    down_genes = down[gene_col].tolist()

    up_available = [g for g in up_genes if g in expr.columns]
    down_available = [g for g in down_genes if g in expr.columns]

    print(f"UP genes available: {len(up_available)}/{len(up_genes)}")
    print(f"DOWN genes available: {len(down_available)}/{len(down_genes)}")

    if not up_available or not down_available:
        raise ValueError("No usable up- or downregulated genes were found in DepMap.")

    x = expr.loc[
        expr.index.intersection(set(model_ids)),
        up_available + down_available,
    ].apply(pd.to_numeric, errors="coerce")

    
    x_z = x.apply(lambda col: zscore(col, nan_policy="omit"), axis=0)

    score = x_z[up_available].mean(axis=1) - x_z[down_available].mean(axis=1)
    score_df = score.rename("MSS_TMBH_score").reset_index()
    score_df.columns = ["DepMap_ID", "MSS_TMBH_score"]

    return score_df, up_available, down_available


In [ ]:


CRC_ONCOTREE_CODES = ["COAD", "READ", "MACR", "COADREAD"]

crc_models = models[
    models["OncotreeCode"].isin(CRC_ONCOTREE_CODES)
].copy()

print("Broad CRC models:", len(crc_models))
display(
    crc_models["OncotreeCode"]
    .value_counts(dropna=False)
    .rename_axis("OncotreeCode")
    .reset_index(name="n_models")
)


In [ ]:




UTERINE_ONCOTREE_CODES = [
    "UCEC", "UCS", "UCCC", "USARC", "ESS", "ULMS", "UASC"
]

ucec_models = models[
    models["OncotreeCode"].isin(UTERINE_ONCOTREE_CODES)
].copy()

print("Broad uterine models:", len(ucec_models))
display(
    ucec_models["OncotreeCode"]
    .value_counts(dropna=False)
    .rename_axis("OncotreeCode")
    .reset_index(name="n_models")
)


UTERINE_CARCINOMA_CODES = ["UCEC", "UCS", "UCCC", "UASC"]
ucec_carcinoma_models = models[
    models["OncotreeCode"].isin(UTERINE_CARCINOMA_CODES)
].copy()

print("Uterine carcinoma-focused models:", len(ucec_carcinoma_models))


In [ ]:
de_crc = pd.read_csv(
    "data/DE_results/MSI_H_vs_MSS_TMB_H.csv", 
    index_col=0
)
de_crc["gene"] = de_crc.index.astype(str).str.upper()
de_crc["log2FoldChange"] = de_crc["logFC"].astype(float)
de_crc["padj"] = de_crc["FDR"].astype(float)
de_crc

In [ ]:
de_ucec = pd.read_csv(
    "data/DE_results/ucec/MSI_H_vs_MSS_TMB_H.csv", 
    index_col=0
)
de_ucec["gene"] = de_ucec.index.astype(str).str.upper()
de_ucec["log2FoldChange"] = de_ucec["logFC"].astype(float)
de_ucec["padj"] = de_ucec["FDR"].astype(float)
de_ucec

In [ ]:
expr.index = expr["MODELID"]
expr

In [ ]:
signature_excluded_genes = sorted({
    gene
    for config in treatment_hypotheses.values()
    for gene in config["targets"]
} | {"SLC3A2", "GPX4", "GCLC", "GCLM", "NFE2L2", "KEAP1", "AIFM2", "ACSL4", "TFRC", "GOT1"})

crc_score, crc_up, crc_down = build_mss_tmbh_score(
    expr=expr,
    model_ids=crc_models["ModelID"],
    de=de_crc,
    top_n=50,
    exclude_genes=signature_excluded_genes,
)

crc_score.head()


In [ ]:
ucec_score, ucec_up, ucec_down = build_mss_tmbh_score(
    expr=expr,
    model_ids=ucec_models["ModelID"],
    de=de_ucec,
    top_n=50,
    exclude_genes=signature_excluded_genes,
)

ucec_score.head()


In [ ]:
def test_signature_drug_response(
    score_df,
    prism,
    treatment_hypotheses,
    response_col="auc",
    min_n=5,
):
    """Correlate the MSS/TMB-H score with PRISM drug response.

    For AUC, rho < 0 means higher MSS/TMB-H similarity is associated with
    greater drug sensitivity.
    """
    results = []

    for hypothesis, config in treatment_hypotheses.items():
        for requested_drug in config["drugs"]:
            drug_df = get_drug_response(
                prism=prism,
                requested_drug=requested_drug,
                response_col=response_col,
            )

            if drug_df.empty:
                continue

            df = score_df.merge(
                drug_df[["depmap_id", "name", response_col]],
                left_on="DepMap_ID",
                right_on="depmap_id",
                how="inner",
            )
            df = df.replace([np.inf, -np.inf], np.nan)
            df = df.dropna(subset=["MSS_TMBH_score", response_col])

            if len(df) < min_n:
                continue

            rho, p = spearmanr(df["MSS_TMBH_score"], df[response_col])
            results.append({
                "hypothesis": hypothesis,
                "drug": drug_df["name"].iloc[0],
                "analysis": "MSS/TMB-H signature",
                "n": df["DepMap_ID"].nunique(),
                "rho": rho,
                "pvalue": p,
            })

    results = pd.DataFrame(results)
    if not results.empty:
        results["FDR_global"] = multipletests(results["pvalue"], method="fdr_bh")[1]
        results["FDR_within_hypothesis"] = (
            results.groupby("hypothesis")["pvalue"]
            .transform(lambda p: multipletests(p, method="fdr_bh")[1])
        )
        results = results.sort_values("pvalue")

    return results


def test_target_expression_drug_response(
    expr,
    model_ids,
    prism,
    treatment_hypotheses,
    response_col="auc",
    min_n=5,
):
    """Test whether expression of each proposed target predicts response.

    For AUC, rho < 0 means higher target expression is associated with
    greater drug sensitivity.
    """
    results = []
    model_ids = set(model_ids)

    for hypothesis, config in treatment_hypotheses.items():
        for target in config["targets"]:
            if target not in expr.columns:
                print(f"Expression unavailable: {target}")
                continue

            target_expr = (
                expr.loc[expr.index.intersection(model_ids), [target]]
                .reset_index()
                .rename(columns={expr.index.name or "index": "DepMap_ID"})
            )
            if "DepMap_ID" not in target_expr.columns:
                target_expr = target_expr.rename(columns={target_expr.columns[0]: "DepMap_ID"})

            for requested_drug in config["drugs"]:
                drug_df = get_drug_response(
                    prism=prism,
                    requested_drug=requested_drug,
                    response_col=response_col,
                )
                if drug_df.empty:
                    continue

                df = target_expr.merge(
                    drug_df[["depmap_id", "name", response_col]],
                    left_on="DepMap_ID",
                    right_on="depmap_id",
                    how="inner",
                )
                df = df.replace([np.inf, -np.inf], np.nan)
                df[target] = pd.to_numeric(df[target], errors="coerce")
                df = df.dropna(subset=[target, response_col])

                if len(df) < min_n:
                    continue

                rho, p = spearmanr(df[target], df[response_col])
                results.append({
                    "hypothesis": hypothesis,
                    "target": target,
                    "drug": drug_df["name"].iloc[0],
                    "analysis": "target expression",
                    "n": df["DepMap_ID"].nunique(),
                    "rho": rho,
                    "pvalue": p,
                })

    results = pd.DataFrame(results)
    if not results.empty:
        results["FDR_global"] = multipletests(results["pvalue"], method="fdr_bh")[1]
        results["FDR_within_hypothesis"] = (
            results.groupby("hypothesis")["pvalue"]
            .transform(lambda p: multipletests(p, method="fdr_bh")[1])
        )
        results = results.sort_values("pvalue")

    return results


In [ ]:
crc_signature_results = test_signature_drug_response(
    score_df=crc_score,
    prism=prism,
    treatment_hypotheses=treatment_hypotheses,
)

crc_target_results = test_target_expression_drug_response(
    expr=expr,
    model_ids=crc_models["ModelID"],
    prism=prism,
    treatment_hypotheses=treatment_hypotheses,
)

crc_signature_results


In [ ]:
ucec_signature_results = test_signature_drug_response(
    score_df=ucec_score,
    prism=prism,
    treatment_hypotheses=treatment_hypotheses,
)

ucec_target_results = test_target_expression_drug_response(
    expr=expr,
    model_ids=ucec_models["ModelID"],
    prism=prism,
    treatment_hypotheses=treatment_hypotheses,
)

ucec_signature_results


In [ ]:
def add_cancer(df, cancer):
    out = df.copy()
    out.insert(0, "cancer", cancer)
    return out

all_signature_results = pd.concat(
    [
        add_cancer(crc_signature_results, "CRC"),
        add_cancer(ucec_signature_results, "UCEC"),
    ],
    ignore_index=True,
)

all_target_results = pd.concat(
    [
        add_cancer(crc_target_results, "CRC"),
        add_cancer(ucec_target_results, "UCEC"),
    ],
    ignore_index=True,
)

print("Signature-level tests")
display(all_signature_results.sort_values(["cancer", "pvalue"]))

print("Direct target-expression tests")
display(all_target_results.sort_values(["cancer", "pvalue"]))

all_signature_results.to_csv("all_signature_drug_results.csv", index=False)
all_target_results.to_csv("all_target_expression_drug_results.csv", index=False)
drug_availability.to_csv("prism_drug_availability.csv", index=False)


In [ ]:
robustness_results = []

for cancer, cancer_models, de in [
    ("CRC", crc_models, de_crc),
    ("UCEC", ucec_models, de_ucec),
]:
    for top_n in [20, 50, 100]:
        score_df, _, _ = build_mss_tmbh_score(
            expr=expr,
            model_ids=cancer_models["ModelID"],
            de=de,
            top_n=top_n,
            exclude_genes=signature_excluded_genes,
        )
        tmp = test_signature_drug_response(
            score_df=score_df,
            prism=prism,
            treatment_hypotheses=treatment_hypotheses,
        )
        tmp.insert(0, "cancer", cancer)
        tmp["top_n"] = top_n
        robustness_results.append(tmp)

robustness_results = pd.concat(robustness_results, ignore_index=True)
robustness_results.sort_values(["hypothesis", "drug", "cancer", "top_n"])
robustness_results.to_csv("signature_drug_results_robustness.csv", index=False)



In [ ]:
def test_crispr_dependency(score_df, gene_effect, gene="GOT1", min_n=5):
    """Correlate MSS/TMB-H score with a DepMap CRISPR gene-effect column."""
    if gene not in gene_effect.columns:
        raise KeyError(f"{gene} not found in gene-effect matrix")

    dep = gene_effect[[gene]].reset_index()
    dep.columns = ["DepMap_ID", "gene_effect"]
    df = score_df.merge(dep, on="DepMap_ID", how="inner").dropna()

    if len(df) < min_n:
        raise ValueError(f"Too few matched models: n={len(df)}")

    rho, p = spearmanr(df["MSS_TMBH_score"], df["gene_effect"])
    return pd.DataFrame([{
        "gene": gene,
        "n": df["DepMap_ID"].nunique(),
        "rho": rho,
        "pvalue": p,
        "interpretation": "rho < 0 means higher MSS/TMB-H score is associated with stronger dependency",
    }])








In [ ]:

def model_coverage_summary(label, selected_models, expr, score_df):
    selected_ids = set(selected_models["ModelID"].dropna())
    expression_ids = selected_ids.intersection(set(expr.index))
    score_ids = selected_ids.intersection(set(score_df["DepMap_ID"]))

    return {
        "cohort": label,
        "selected_models": len(selected_ids),
        "with_expression": len(expression_ids),
        "with_signature_score": len(score_ids),
    }


coverage_summary = pd.DataFrame([
    model_coverage_summary("CRC broad", crc_models, expr, crc_score),
    model_coverage_summary("Uterine broad", ucec_models, expr, ucec_score),
])

display(coverage_summary)
coverage_summary.to_csv("model_coverage_summary.csv", index=False)


In [ ]:

def drug_coverage_diagnostics(
    label,
    selected_models,
    score_df,
    prism,
    treatment_hypotheses,
    response_col="auc",
):
    selected_ids = set(selected_models["ModelID"].dropna())
    expression_ids = selected_ids.intersection(set(expr.index))
    score_ids = set(score_df["DepMap_ID"].dropna())
    rows = []

    for hypothesis, config in treatment_hypotheses.items():
        for requested_drug in config["drugs"]:
            drug_df = get_drug_response(
                prism=prism,
                requested_drug=requested_drug,
                response_col=response_col,
            )

            if drug_df.empty:
                rows.append({
                    "cohort": label,
                    "hypothesis": hypothesis,
                    "drug": requested_drug,
                    "available": False,
                    "selected_models": len(selected_ids),
                    "with_expression": len(expression_ids),
                    "with_score": len(score_ids),
                    "with_drug_response_all_lineages": 0,
                    "final_signature_n": 0,
                })
                continue

            drug_ids = set(
                drug_df.loc[drug_df[response_col].notna(), "depmap_id"].dropna()
            )

            rows.append({
                "cohort": label,
                "hypothesis": hypothesis,
                "drug": drug_df["name"].iloc[0],
                "available": True,
                "selected_models": len(selected_ids),
                "with_expression": len(expression_ids),
                "with_score": len(score_ids),
                "with_drug_response_all_lineages": len(drug_ids),
                "final_signature_n": len(score_ids.intersection(drug_ids)),
            })

    return pd.DataFrame(rows)


coverage_by_drug = pd.concat([
    drug_coverage_diagnostics(
        "CRC broad", crc_models, crc_score, prism, treatment_hypotheses
    ),
    drug_coverage_diagnostics(
        "Uterine broad", ucec_models, ucec_score, prism, treatment_hypotheses
    ),
], ignore_index=True)

display(
    coverage_by_drug[
        coverage_by_drug["available"]
    ].sort_values(["cohort", "final_signature_n", "hypothesis"])
)

coverage_by_drug.to_csv("drug_coverage_diagnostics.csv", index=False)


In [ ]:

ucec_carcinoma_score, ucec_carcinoma_up, ucec_carcinoma_down = build_mss_tmbh_score(
    expr=expr,
    model_ids=ucec_carcinoma_models["ModelID"],
    de=de_ucec,
    top_n=50,
    exclude_genes=signature_excluded_genes,
)

ucec_carcinoma_signature_results = test_signature_drug_response(
    score_df=ucec_carcinoma_score,
    prism=prism,
    treatment_hypotheses=treatment_hypotheses,
)

ucec_carcinoma_target_results = test_target_expression_drug_response(
    expr=expr,
    model_ids=ucec_carcinoma_models["ModelID"],
    prism=prism,
    treatment_hypotheses=treatment_hypotheses,
)

display(ucec_carcinoma_signature_results)
display(ucec_carcinoma_target_results)

ucec_carcinoma_signature_results.to_csv(
    "uterine_carcinoma_signature_drug_results.csv", index=False
)
ucec_carcinoma_target_results.to_csv(
    "uterine_carcinoma_target_expression_drug_results.csv", index=False
)


In [ ]:

def build_drug_class_response(
    prism,
    treatment_hypotheses,
    response_col="auc",
    min_drugs_per_model=1,
):
    pieces = []

    for hypothesis, config in treatment_hypotheses.items():
        requested = {d.lower() for d in config["drugs"]}
        tmp = prism[
            prism["name"].astype(str).str.lower().isin(requested)
        ][["depmap_id", "name", response_col]].copy()

        if tmp.empty:
            continue

        tmp[response_col] = pd.to_numeric(tmp[response_col], errors="coerce")
        tmp = tmp.dropna(subset=["depmap_id", response_col])

        
        tmp["response_z"] = tmp.groupby("name")[response_col].transform(
            lambda x: (x - x.mean()) / x.std(ddof=0)
            if x.std(ddof=0) > 0 else np.nan
        )
        tmp["hypothesis"] = hypothesis
        pieces.append(tmp)

    if not pieces:
        return pd.DataFrame()

    long_df = pd.concat(pieces, ignore_index=True)

    class_response = (
        long_df.groupby(["hypothesis", "depmap_id"], as_index=False)
        .agg(
            class_response_z=("response_z", "mean"),
            n_drugs_measured=("name", "nunique"),
            drugs_measured=("name", lambda x: "; ".join(sorted(set(x)))),
        )
    )

    return class_response[
        class_response["n_drugs_measured"] >= min_drugs_per_model
    ].copy()


def test_signature_drug_classes(
    score_df,
    class_response,
    min_n=5,
):
    rows = []

    for hypothesis, group in class_response.groupby("hypothesis"):
        df = score_df.merge(
            group,
            left_on="DepMap_ID",
            right_on="depmap_id",
            how="inner",
        ).dropna(subset=["MSS_TMBH_score", "class_response_z"])

        if len(df) < min_n:
            continue

        rho, p = spearmanr(
            df["MSS_TMBH_score"],
            df["class_response_z"],
        )

        rows.append({
            "hypothesis": hypothesis,
            "analysis": "drug-class response",
            "n": df["DepMap_ID"].nunique(),
            "median_number_of_drugs_per_model": df["n_drugs_measured"].median(),
            "rho": rho,
            "pvalue": p,
        })

    out = pd.DataFrame(rows)

    if not out.empty:
        out["FDR_global"] = multipletests(
            out["pvalue"], method="fdr_bh"
        )[1]
        out = out.sort_values("pvalue")

    return out


drug_class_response = build_drug_class_response(
    prism=prism,
    treatment_hypotheses=treatment_hypotheses,
)

crc_class_results = test_signature_drug_classes(
    crc_score, drug_class_response
)
ucec_class_results = test_signature_drug_classes(
    ucec_score, drug_class_response
)
ucec_carcinoma_class_results = test_signature_drug_classes(
    ucec_carcinoma_score, drug_class_response
)

all_class_results = pd.concat([
    add_cancer(crc_class_results, "CRC broad"),
    add_cancer(ucec_class_results, "Uterine broad"),
    add_cancer(ucec_carcinoma_class_results, "Uterine carcinoma-focused"),
], ignore_index=True)

display(all_class_results)
all_class_results.to_csv("drug_class_signature_results.csv", index=False)


In [ ]:

def resolve_lineage_column(models):
    candidates = [
        "OncotreeLineage",
        "OncotreePrimaryDisease",
        "OncotreeSubtype",
    ]
    for col in candidates:
        if col in models.columns:
            return col
    raise KeyError(
        "No lineage column found. Checked: " + ", ".join(candidates)
    )


LINEAGE_COLUMN = resolve_lineage_column(models)
print("Lineage adjustment column:", LINEAGE_COLUMN)


def fit_lineage_adjusted_drug_models(
    score_df,
    prism,
    models,
    treatment_hypotheses,
    response_col="auc",
    min_n=30,
    min_models_per_lineage=3,
):
    model_meta = models[
        ["ModelID", LINEAGE_COLUMN]
    ].drop_duplicates("ModelID").rename(
        columns={
            "ModelID": "DepMap_ID",
            LINEAGE_COLUMN: "lineage",
        }
    )

    rows = []

    for hypothesis, config in treatment_hypotheses.items():
        for requested_drug in config["drugs"]:
            drug_df = prism[
                prism["name"].astype(str).str.lower()
                == requested_drug.lower()
            ][["depmap_id", "name", response_col]].copy()

            if drug_df.empty:
                continue

            df = (
                score_df
                .merge(model_meta, on="DepMap_ID", how="inner")
                .merge(
                    drug_df,
                    left_on="DepMap_ID",
                    right_on="depmap_id",
                    how="inner",
                )
            )

            df[response_col] = pd.to_numeric(
                df[response_col], errors="coerce"
            )
            df = df.dropna(
                subset=["MSS_TMBH_score", response_col, "lineage"]
            )

            lineage_counts = df["lineage"].value_counts()
            retained_lineages = lineage_counts[
                lineage_counts >= min_models_per_lineage
            ].index
            df = df[df["lineage"].isin(retained_lineages)].copy()

            if len(df) < min_n or df["lineage"].nunique() < 2:
                continue

            try:
                fit = smf.ols(
                    f"{response_col} ~ MSS_TMBH_score + C(lineage)",
                    data=df,
                ).fit(cov_type="HC3")

                rows.append({
                    "hypothesis": hypothesis,
                    "drug": drug_df["name"].iloc[0],
                    "analysis": "pan-cancer lineage-adjusted OLS",
                    "n": df["DepMap_ID"].nunique(),
                    "n_lineages": df["lineage"].nunique(),
                    "beta_score": fit.params["MSS_TMBH_score"],
                    "pvalue": fit.pvalues["MSS_TMBH_score"],
                    "ci_low": fit.conf_int().loc[
                        "MSS_TMBH_score", 0
                    ],
                    "ci_high": fit.conf_int().loc[
                        "MSS_TMBH_score", 1
                    ],
                })
            except Exception as exc:
                print(
                    f"Skipped {hypothesis} / {requested_drug}: {exc}"
                )

    out = pd.DataFrame(rows)

    if not out.empty:
        out["FDR_global"] = multipletests(
            out["pvalue"], method="fdr_bh"
        )[1]
        out["FDR_within_hypothesis"] = (
            out.groupby("hypothesis")["pvalue"]
            .transform(
                lambda p: multipletests(
                    p, method="fdr_bh"
                )[1]
            )
        )
        out = out.sort_values("pvalue")

    return out


In [ ]:


all_model_ids = models["ModelID"].dropna().unique()

crc_score_all_models, _, _ = build_mss_tmbh_score(
    expr=expr,
    model_ids=all_model_ids,
    de=de_crc,
    top_n=50,
    exclude_genes=signature_excluded_genes,
)

ucec_score_all_models, _, _ = build_mss_tmbh_score(
    expr=expr,
    model_ids=all_model_ids,
    de=de_ucec,
    top_n=50,
    exclude_genes=signature_excluded_genes,
)

crc_pan_cancer_results = fit_lineage_adjusted_drug_models(
    score_df=crc_score_all_models,
    prism=prism,
    models=models,
    treatment_hypotheses=treatment_hypotheses,
)

ucec_pan_cancer_results = fit_lineage_adjusted_drug_models(
    score_df=ucec_score_all_models,
    prism=prism,
    models=models,
    treatment_hypotheses=treatment_hypotheses,
)

crc_pan_cancer_results.insert(0, "signature_source", "CRC")
ucec_pan_cancer_results.insert(0, "signature_source", "UCEC")

all_pan_cancer_results = pd.concat([
    crc_pan_cancer_results,
    ucec_pan_cancer_results,
], ignore_index=True)

display(all_pan_cancer_results)
all_pan_cancer_results.to_csv(
    "pan_cancer_lineage_adjusted_results.csv",
    index=False,
)


In [ ]:

def prepare_external_response(
    response_df,
    model_id_col,
    drug_col,
    response_col,
    model_id_map=None,
):
    out = response_df[
        [model_id_col, drug_col, response_col]
    ].copy()

    out.columns = ["external_model_id", "name", "auc"]

    if model_id_map is not None:
        required = {"external_model_id", "DepMap_ID"}
        missing = required.difference(model_id_map.columns)
        if missing:
            raise KeyError(
                f"model_id_map is missing columns: {sorted(missing)}"
            )

        out = out.merge(
            model_id_map[
                ["external_model_id", "DepMap_ID"]
            ].drop_duplicates(),
            on="external_model_id",
            how="inner",
        )
        out["depmap_id"] = out["DepMap_ID"]
    else:
        
        out["depmap_id"] = out["external_model_id"]

    out["name"] = out["name"].astype(str)
    out["auc"] = pd.to_numeric(out["auc"], errors="coerce")
    return out.dropna(subset=["depmap_id", "name", "auc"])



















In [ ]:

DEPENDENCY_GENES = {
    "GOT1/metabolism": ["GOT1"],
    "SLC7A11/ferroptosis": [
        "SLC7A11", "GPX4", "GCLC", "GCLM", "SLC3A2"
    ],
    "PI3K/AKT/mTOR": [
        "PIK3CA", "PIK3R1", "AKT1", "MTOR", "RPTOR"
    ],
    "replication stress": [
        "WEE1", "CHEK1", "ATR", "CDC7", "CDK1"
    ],
    "CDK4/6": ["CDK4", "CDK6", "RB1"],
}


def test_crispr_dependency_panel(
    score_df,
    gene_effect,
    dependency_genes=DEPENDENCY_GENES,
    min_n=5,
):
    rows = []

    
    ge = gene_effect.copy()
    ge.index = ge.index.astype(str)

    for pathway, genes in dependency_genes.items():
        for gene in genes:
            if gene not in ge.columns:
                continue

            dep = ge[[gene]].reset_index()
            dep.columns = ["DepMap_ID", "gene_effect"]
            dep["gene_effect"] = pd.to_numeric(
                dep["gene_effect"], errors="coerce"
            )

            df = score_df.merge(
                dep, on="DepMap_ID", how="inner"
            ).dropna(
                subset=["MSS_TMBH_score", "gene_effect"]
            )

            if len(df) < min_n:
                continue

            rho, p = spearmanr(
                df["MSS_TMBH_score"],
                df["gene_effect"],
            )

            rows.append({
                "pathway": pathway,
                "gene": gene,
                "n": df["DepMap_ID"].nunique(),
                "rho": rho,
                "pvalue": p,
            })

    out = pd.DataFrame(rows)

    if not out.empty:
        out["FDR_global"] = multipletests(
            out["pvalue"], method="fdr_bh"
        )[1]
        out["FDR_within_pathway"] = (
            out.groupby("pathway")["pvalue"]
            .transform(
                lambda p: multipletests(
                    p, method="fdr_bh"
                )[1]
            )
        )
        out = out.sort_values("pvalue")

    return out















